# 🏗️ PHASE 2: COMPLEX LAYOUTS (SUPREME THEME)
## Focus: 3-Pillar Grid & Snake Process Flow

In [ ]:
!pip install python-pptx pydantic -q

In [ ]:
import json
import base64
import os
import math
from typing import List, Optional, Literal, Union, Dict, Any
from pydantic import BaseModel
from pptx import Presentation
from pptx.util import Inches, Pt
from pptx.enum.text import PP_ALIGN, MSO_ANCHOR
from pptx.dml.color import RGBColor
from pptx.enum.shapes import MSO_SHAPE, MSO_CONNECTOR
from IPython.display import display, HTML

# ==========================================
# 🎨 1. THE SUPREME DESIGN SYSTEM
# ==========================================
class Colors:
    BG = [245, 245, 245]
    TITLE = [0, 0, 0]
    TEXT = [60, 60, 60]
    ACCENT_BANNER = [225, 226, 246] # Lavender
    ACCENT_PRIMARY = [0, 51, 102]   # Navy (Icons/Headers)
    ACCENT_SECONDARY = [200, 200, 200] # Borders
    PROCESS_NODE = [255, 255, 255]
    PROCESS_BORDER = [0, 112, 192]

class Fonts:
    SERIF = "Times New Roman"
    SANS = "Arial"

def get_rgb(c): return RGBColor(c[0], c[1], c[2])

# ==========================================
# 📝 2. DATA MODELS
# ==========================================

class SlideCover(BaseModel):
    type: Literal["cover"]
    title: str
    subtitle: str
    version_date: str

class SlideColumns(BaseModel):
    type: Literal["columns"]
    title: str
    banner: str
    columns: List[Dict[str, Any]] # title, items, icon

class SlideProcess(BaseModel):
    type: Literal["process"]
    title: str
    steps: List[Dict[str, str]] # title, desc

class PresentationConfig(BaseModel):
    filename: str
    slides: List[Union[SlideCover, SlideColumns, SlideProcess]]

# ==========================================
# 🖌️ 3. RENDERER ENGINE
# ==========================================

def add_header(slide, title):
    # Standard Supreme Header
    tb = slide.shapes.add_textbox(Inches(0.5), Inches(0.4), Inches(12), Inches(1))
    p = tb.text_frame.paragraphs[0]
    p.text = title
    p.font.name = Fonts.SERIF
    p.font.bold = True
    p.font.size = Pt(36)
    p.font.color.rgb = get_rgb(Colors.TITLE)

def render_cover(prs, data: SlideCover):
    slide = prs.slides.add_slide(prs.slide_layouts[6])
    slide.background.fill.solid()
    slide.background.fill.fore_color.rgb = get_rgb(Colors.BG)
    
    # Logo Placeholders
    slide.shapes.add_shape(MSO_SHAPE.RECTANGLE, Inches(0.5), Inches(2), Inches(1.5), Inches(0.8)).text_frame.text = "LOGO 1"
    slide.shapes.add_shape(MSO_SHAPE.RECTANGLE, Inches(11), Inches(0.5), Inches(1.5), Inches(0.6)).text_frame.text = "LOGO 2"

    tb = slide.shapes.add_textbox(Inches(0.5), Inches(4), Inches(12), Inches(2))
    p = tb.text_frame.paragraphs[0]
    p.text = data.title
    p.font.name = Fonts.SERIF
    p.font.size = Pt(44)
    p.font.bold = True

    sb = slide.shapes.add_textbox(Inches(0.5), Inches(5.5), Inches(12), Inches(1))
    p = sb.text_frame.paragraphs[0]
    p.text = f"{data.subtitle}\n{data.version_date}"
    p.font.name = Fonts.SERIF
    p.font.size = Pt(16)

def render_columns(prs, data: SlideColumns):
    slide = prs.slides.add_slide(prs.slide_layouts[6])
    slide.background.fill.solid()
    slide.background.fill.fore_color.rgb = get_rgb(Colors.BG)
    
    add_header(slide, data.title)

    # Banner Box
    box = slide.shapes.add_shape(MSO_SHAPE.RECTANGLE, Inches(0.5), Inches(1.3), Inches(12.33), Inches(0.6))
    box.fill.solid()
    box.fill.fore_color.rgb = get_rgb(Colors.ACCENT_BANNER)
    box.line.color.rgb = get_rgb(Colors.ACCENT_SECONDARY)
    p = box.text_frame.paragraphs[0]
    p.text = data.banner
    p.font.name = Fonts.SANS
    p.font.size = Pt(11)
    p.font.color.rgb = get_rgb(Colors.TEXT)
    p.alignment = PP_ALIGN.LEFT
    box.text_frame.margin_left = Inches(0.1)

    # Columns Calculation
    col_count = len(data.columns)
    margin = 0.5
    gap = 0.3
    col_width = (13.33 - (2 * margin) - ((col_count - 1) * gap)) / col_count
    start_y = 2.2

    for i, col in enumerate(data.columns):
        left = margin + (i * (col_width + gap))
        
        # 1. Icon Placeholder (Circle)
        icon = slide.shapes.add_shape(MSO_SHAPE.OVAL, left, start_y, Inches(0.6), Inches(0.6))
        icon.fill.solid()
        icon.fill.fore_color.rgb = get_rgb([255, 255, 255])
        icon.line.color.rgb = get_rgb(Colors.ACCENT_PRIMARY)
        icon.line.width = Pt(2)
        # Placeholder Text inside icon
        p = icon.text_frame.paragraphs[0]
        p.text = col.get("icon_name", "Icon")
        p.font.size = Pt(8)
        p.font.color.rgb = get_rgb(Colors.ACCENT_PRIMARY)

        # 2. Title (Serif, Bold)
        # Positioned right of icon or below? Image implies right of icon for header
        # Let's put title below icon to be safe on spacing
        tb = slide.shapes.add_textbox(left + Inches(0.7), start_y, col_width - Inches(0.7), Inches(0.8))
        tb.text_frame.word_wrap = True
        p = tb.text_frame.paragraphs[0]
        p.text = col['title']
        p.font.name = Fonts.SERIF
        p.font.bold = True
        p.font.size = Pt(16)
        p.font.color.rgb = get_rgb(Colors.ACCENT_PRIMARY)
        
        # 3. Bullet Points
        bb = slide.shapes.add_textbox(left, start_y + Inches(0.8), col_width, Inches(4))
        tf = bb.text_frame
        tf.word_wrap = True
        
        for item in col['items']:
            p = tf.add_paragraph()
            p.text = item
            p.font.name = Fonts.SANS
            p.font.size = Pt(11)
            p.font.color.rgb = get_rgb(Colors.TEXT)
            p.space_after = Pt(6)
            p.level = 0

def render_process(prs, data: SlideProcess):
    # Renders a "Zig-Zag" Snake flow to approximate the S-Curve image
    slide = prs.slides.add_slide(prs.slide_layouts[6])
    slide.background.fill.solid()
    slide.background.fill.fore_color.rgb = get_rgb(Colors.BG)
    
    add_header(slide, data.title)

    steps = data.steps
    # Hardcoded coordinates for a 5-step zig-zag flow
    # 1 -> 2 -> 3
    #           |
    # 5 <- 4 <--
    
    positions = [
        (1.5, 3.0), (5.5, 3.0), (9.5, 3.0), # Row 1
        (9.5, 5.5), (5.5, 5.5)              # Row 2 (Reversed)
    ]
    
    prev_node = None
    
    for i, step in enumerate(steps):
        if i >= len(positions): break
        
        x, y = positions[i]
        x_in = Inches(x)
        y_in = Inches(y)
        size = Inches(1.2)
        
        # 1. Draw Connector from Previous
        if prev_node:
            conn = slide.shapes.add_connector(MSO_CONNECTOR.ELBOW, 
                                              prev_node.left + (size/2), prev_node.top + (size/2),
                                              x_in + (size/2), y_in + (size/2))
            conn.line.color.rgb = get_rgb(Colors.ACCENT_SECONDARY)
            conn.line.width = Pt(3)
            # Send to back hack (add first then nodes)
        
        # 2. Draw Node Circle
        node = slide.shapes.add_shape(MSO_SHAPE.OVAL, x_in, y_in, size, size)
        node.fill.solid()
        node.fill.fore_color.rgb = get_rgb(Colors.PROCESS_NODE)
        node.line.color.rgb = get_rgb(Colors.PROCESS_BORDER)
        node.line.width = Pt(3)
        
        # Icon/Number inside
        p = node.text_frame.paragraphs[0]
        p.text = str(i + 1)
        p.font.bold = True
        p.font.size = Pt(24)
        p.font.color.rgb = get_rgb(Colors.PROCESS_BORDER)
        p.alignment = PP_ALIGN.CENTER
        
        prev_node = node
        
        # 3. Text Descriptions
        # Logic: If row 1, text above. If row 2, text below.
        text_y = y_in - Inches(1.2) if i < 3 else y_in + Inches(1.3)
        
        tb = slide.shapes.add_textbox(x_in - Inches(0.9), text_y, Inches(3), Inches(1))
        p = tb.text_frame.paragraphs[0]
        p.text = step['title']
        p.font.name = Fonts.SERIF
        p.font.bold = True
        p.font.size = Pt(12)
        p.alignment = PP_ALIGN.CENTER
        
        p = tb.text_frame.add_paragraph()
        p.text = step['desc']
        p.font.name = Fonts.SANS
        p.font.size = Pt(10)
        p.alignment = PP_ALIGN.CENTER

# ==========================================
# 🚀 4. USER JSON INPUT
# ==========================================

input_json = {
  "filename": "Supreme_Phase2.pptx",
  "slides": [
    # SLIDE 1: COVER (Standard)
    {
      "type": "cover",
      "title": "Project Report - Biller Management Portal",
      "subtitle": "Phase 2 Update: Unified Platform & Process",
      "version_date": "Version 1.2 | December 11, 2025"
    },
    # SLIDE 2: 3-PILLAR LAYOUT (From Image 1)
    {
      "type": "columns",
      "title": "Our Solution: A Unified Three Pillar Platform",
      "banner": "Our solution is a single, unified platform built on three distinct but interconnected pillars that work together to create a comprehensive biller management ecosystem.",
      "columns": [
        {
          "title": "Administrator Portal: Control & Oversight",
          "icon_name": "Admin",
          "items": [
            "Full Biller Lifecycle Management: Onboarding wizard, comprehensive dashboard.",
            "Advanced User & Access Control: Secure RBAC system.",
            "Platform Analytics: Real-time KPIs and visual charts."
          ]
        },
        {
          "title": "Biller Portal: Empowerment & Efficiency",
          "icon_name": "Biller",
          "items": [
            "Personalized Dashboard: KPI-driven dashboard with activity feed.",
            "Customer & Bill Management: Full CRUD operations.",
            "Financial Visibility: Master Payment History page."
          ]
        },
        {
          "title": "Partner API Gateway: Integration & Growth",
          "icon_name": "API",
          "items": [
            "Secure & Authenticated: Hashed unique API keys.",
            "Core Capabilities: Endpoints for validating customers.",
            "Future-Ready: Versioned (/v1) and documented."
          ]
        }
      ]
    },
    # SLIDE 3: SNAKE PROCESS (From Image 2)
    {
      "type": "process",
      "title": "Cycle Diagrams Theme",
      "steps": [
        {"title": "Step 1", "desc": "Mars is red but actually a cold place."},
        {"title": "Step 2", "desc": "Earth is the third planet from the Sun."},
        {"title": "Step 3", "desc": "Saturn is a gas giant with rings."},
        {"title": "Step 4", "desc": "Jupiter is the largest planet."},
        {"title": "Step 5", "desc": "Mercury is the smallest planet."}
      ]
    }
  ]
}

# ==========================================
# 🏁 5. EXECUTE
# ==========================================

try:
    print("⚙️ Generating Phase 2 Deck...")
    config = PresentationConfig(**input_json)
    prs = Presentation()
    prs.slide_width = Inches(13.333)
    prs.slide_height = Inches(7.5)

    for slide_data in config.slides:
        if slide_data.type == "cover":
            render_cover(prs, slide_data)
        elif slide_data.type == "columns":
            render_columns(prs, slide_data)
        elif slide_data.type == "process":
            render_process(prs, slide_data)

    prs.save(config.filename)
    
    with open(config.filename, "rb") as f:
        b64 = base64.b64encode(f.read()).decode()
    
    link = f'<a href="data:application/vnd.openxmlformats-officedocument.presentationml.presentation;base64,{b64}" download="{config.filename}" style="background: #0078D4; color: white; padding: 10px 20px; border-radius: 5px; text-decoration: none; font-weight: bold;">⬇️ DOWNLOAD PHASE 2 PPT</a>'
    display(HTML(link))
    print("✅ Done.")

except Exception as e:
    print(f"❌ Error: {e}")